# LLM Fine-tuning（Unsloth / Colab）

**操作を最小にする手順**

1. （初回のみ）Colab 左メニュー **🔑 シークレット** に `HF_TOKEN`（Hugging Face の Write トークン）を登録する。
2. **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ。
3. **ランタイム → すべてのセルを実行**（または上から順に実行）。

事前に `training/params.yaml` の `hf_lora_repo` を自分の Hugging Face リポジトリ名に編集して push しておいてください。プライベート GitHub リポジトリを clone する場合のみ、シークレット `GITHUB_TOKEN` も追加してください（公開リポジトリなら不要）。


## 1. 依存パッケージのインストール


In [ ]:
# Install Unsloth and other dependencies
!pip install -q "unsloth[colab-new]" peft accelerate bitsandbytes transformers trl huggingface_hub pyyaml datasets


## 2. シークレット・リポジトリのクローン

`HF_TOKEN` を環境変数に渡し、学習用コードを `/content/llm-model-generation` に取得します。


In [ ]:
import os
import subprocess
from pathlib import Path


from typing import Optional


def _colab_secret(name: str) -> Optional[str]:
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


REPO_OWNER = "TsutsujiStudyTeam"
REPO_NAME = "llm-model-generation"
CONTENT = Path("/content")
repo_path = CONTENT / REPO_NAME

if repo_path.exists():
    subprocess.run(["rm", "-rf", str(repo_path)], check=False)

os.chdir(CONTENT)

hf_token = _colab_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError(
        "Colab のシークレット HF_TOKEN が未設定です。左メニュー「🔑 シークレット」から追加してください。"
    )
os.environ["HF_TOKEN"] = hf_token

github_token = _colab_secret("GITHUB_TOKEN")
if github_token:
    clone_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
else:
    clone_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

subprocess.run(["git", "clone", clone_url, str(REPO_NAME)], check=True)
os.chdir(repo_path)
print("Working directory:", os.getcwd())


## 3. 学習の実行（`finetune_script.py`）

対話入力はありません。`training/params.yaml` と環境変数 `HF_TOKEN`（および任意の `HF_LORA_REPO`）で学習します。


In [ ]:
!python3 training/finetune_script.py


## 4. （任意）Colab 上の簡易推論スモークテスト

学習済みマージ済みアダプタを `training/lora_adapter_merged` から読み込みます。


In [ ]:
import yaml
from pathlib import Path

import torch
from unsloth import FastLanguageModel

ROOT = Path.cwd()
with open(ROOT / "training" / "params.yaml", encoding="utf-8") as f:
    params = yaml.safe_load(f)
max_seq_length = params["max_seq_length"]

merged = ROOT / "training" / "lora_adapter_merged"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(merged),
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)
messages = [
    {"role": "user", "content": "私は犬が好きです。これを否定文に変換してください。"},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(inputs, max_new_tokens=64, use_cache=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
